In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import sklearn

print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)

TensorFlow: 2.20.0
NumPy: 2.0.2
Pandas: 2.2.2
Scikit-learn: 1.6.1


In [ ]:
import sys
print(sys.version)


3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [ ]:
import nltk
nltk.download("gutenberg")
from nltk.corpus import gutenberg
import numpy as np
import pandas as pd

[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!


In [ ]:
# load datast
data = gutenberg.raw("shakespeare-hamlet.txt")
# save the file
with open('hamlet.txt','w') as file :
  file.write(data)

**Data Preprocessing**

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

In [ ]:
# loading the text
with open('hamlet.txt','r') as file :
  text = file.read().lower()

# Tokenize the text - crating indx for words
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
total_words = len(tokenizer.word_index)+1
total_words

4818

In [ ]:
# create input sequence
input_sequences = []
for line in text.split('\n'):
  token_list = tokenizer.texts_to_sequences([line])[0]
  for i in range(1,len(token_list)) :
    n_gram_sequence = token_list[:i+1]
    input_sequences.append(n_gram_sequence)

In [ ]:
input_sequences

[[1, 687],
 [1, 687, 4],
 [1, 687, 4, 45],
 [1, 687, 4, 45, 41],
 [1, 687, 4, 45, 41, 1886],
 [1, 687, 4, 45, 41, 1886, 1887],
 [1, 687, 4, 45, 41, 1886, 1887, 1888],
 [1180, 1889],
 [1180, 1889, 1890],
 [1180, 1889, 1890, 1891],
 [57, 407],
 [57, 407, 2],
 [57, 407, 2, 1181],
 [57, 407, 2, 1181, 177],
 [57, 407, 2, 1181, 177, 1892],
 [407, 1182],
 [407, 1182, 63],
 [408, 162],
 [408, 162, 377],
 [408, 162, 377, 21],
 [408, 162, 377, 21, 247],
 [408, 162, 377, 21, 247, 882],
 [18, 66],
 [451, 224],
 [451, 224, 248],
 [451, 224, 248, 1],
 [451, 224, 248, 1, 30],
 [408, 407],
 [451, 25],
 [408, 6],
 [408, 6, 43],
 [408, 6, 43, 62],
 [408, 6, 43, 62, 1893],
 [408, 6, 43, 62, 1893, 96],
 [408, 6, 43, 62, 1893, 96, 18],
 [408, 6, 43, 62, 1893, 96, 18, 566],
 [451, 71],
 [451, 71, 51],
 [451, 71, 51, 1894],
 [451, 71, 51, 1894, 567],
 [451, 71, 51, 1894, 567, 378],
 [451, 71, 51, 1894, 567, 378, 80],
 [451, 71, 51, 1894, 567, 378, 80, 3],
 [451, 71, 51, 1894, 567, 378, 80, 3, 273],
 [451, 71

In [ ]:
# pad sequences
max_sequences_len = max([len(x) for x in input_sequences])
max_sequences_len

14

In [ ]:
input_sequences = np.array(pad_sequences(input_sequences,maxlen = max_sequences_len,padding = 'pre'))

In [ ]:
input_sequences

array([[   0,    0,    0, ...,    0,    1,  687],
       [   0,    0,    0, ...,    1,  687,    4],
       [   0,    0,    0, ...,  687,    4,   45],
       ...,
       [   0,    0,    0, ...,    4,   45, 1047],
       [   0,    0,    0, ...,   45, 1047,    4],
       [   0,    0,    0, ..., 1047,    4,  193]], dtype=int32)

In [ ]:
# create predictors and label
import tensorflow as tf
X,y = input_sequences[:,:-1],input_sequences[:,-1]


In [ ]:
X

array([[   0,    0,    0, ...,    0,    0,    1],
       [   0,    0,    0, ...,    0,    1,  687],
       [   0,    0,    0, ...,    1,  687,    4],
       ...,
       [   0,    0,    0, ...,  687,    4,   45],
       [   0,    0,    0, ...,    4,   45, 1047],
       [   0,    0,    0, ...,   45, 1047,    4]], dtype=int32)

In [ ]:
y

array([ 687,    4,   45, ..., 1047,    4,  193], dtype=int32)

In [ ]:
y = tf.keras.utils.to_categorical(y,num_classes=total_words)


In [ ]:
y

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [ ]:
# splitting the data
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2)

**Traning the Model**

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense,Dropout

# define a model
model = Sequential()
model.add(Embedding(total_words,100,input_length= max_sequences_len))
model.add(LSTM(150,return_sequences = True))
model.add(Dropout(0.2))
model.add(LSTM(100))
model.add(Dense(total_words,activation = "softmax"))

# compile the model
model.compile(loss = 'categorical_crossentropy',
              optimizer='adam',metrics=['accuracy'])
model.summary()



/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Train the model
history = model.fit(X_train,y_train,epochs = 50,validation_data =(X_test,y_test),verbose = 1)

Epoch 1/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 55s 75ms/step - accuracy: 0.0331 - loss: 6.8783 - val_accuracy: 0.0352 - val_loss: 6.7260
Epoch 2/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 46s 72ms/step - accuracy: 0.0396 - loss: 6.4450 - val_accuracy: 0.0429 - val_loss: 6.8057
Epoch 3/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 45s 69ms/step - accuracy: 0.0477 - loss: 6.2970 - val_accuracy: 0.0519 - val_loss: 6.8405
Epoch 4/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 44s 69ms/step - accuracy: 0.0522 - loss: 6.1586 - val_accuracy: 0.0554 - val_loss: 6.8975
Epoch 5/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 46s 72ms/step - accuracy: 0.0562 - loss: 6.0278 - val_accuracy: 0.0595 - val_loss: 6.9037
Epoch 6/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 48s 74ms/step - accuracy: 0.0630 - loss: 5.8895 - val_accuracy: 0.0649 - val_loss: 6.9405
Epoch 7/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 48s 74ms/step - accuracy: 0.0685 - loss: 5.7476 - val_accuracy: 0.0666 - val_loss: 7.0118
Epoch 8/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 82s 75ms/step - accuracy: 0.0779 - loss: 5.6030 - 

In [ ]:
# Function to predict the next word
def predict_next_word(model,tokenizr,text,max_sequence_len) :
  token_list = tokenizer.texts_to_sequences([text])[0]
  if len(token_list) >= max_sequence_len :
    token_list = token_list[-(max_sequence_len-1):]
  token_list = pad_sequences([token_list],maxlen = max_sequence_len-1,padding = 'pre')
  predicted = model.predict(token_list,verbose = 0)
  predicted_word_index = np.argmax(predicted,axis = 1)
  for word,index in tokenizer.word_index.items():
    if index == predicted_word_index:
      return word
  return None


In [ ]:
input_text = "To be or not to be"
print(f"Input text:{input_text}")
max_sequence_len = model.input_shape[1]+1
next_word = predict_next_word(model,tokenizer,input_text,max_sequence_len)
print(f"next_word prdiction :{next_word}")

Input text:To be or not to be
next_word prdiction :buried


In [ ]:
# save the model
model.save("next_word_lstm.keras")

#  save the tokenizer
import pickle
with open('tokenizer.pickle','wb') as handle :
  pickle.dump(tokenizer,handle,protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
model.save("next_word_lstm2.keras")

In [ ]:
from google.colab import files

files.download("next_word_lstm2.keras")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files

files.download("next_word_lstm.h5")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>